In [7]:
import json
import os
import ollama

class B2BInsightGenerator:
    def __init__(self, input_file="data/llm_ready_clusters.json", output_file="data/final_b2b_insights.json", model_name="llama3:latest"):
        self.input_file = input_file
        self.output_file = output_file
        self.model_name = model_name
        self.clusters_data = {}

    def load_data(self):
        """Reads the clustered data from the previous pipeline step."""
        if not os.path.exists(self.input_file):
            print(f"[ERROR] Could not find {self.input_file}. Ensure the clustering script ran successfully.")
            return False
            
        print(f"[INFO] Loading data from {self.input_file}...")
        with open(self.input_file, 'r', encoding='utf-8') as f:
            self.clusters_data = json.load(f)
            
        print(f"[INFO] Successfully loaded {len(self.clusters_data)} clusters for analysis.")
        return True

    def generate_insights(self):
        """Passes the data to the local LLM to generate readable business insights."""
        if not self.clusters_data:
            print("[ERROR] No data loaded. Run load_data() first.")
            return

        print(f"[INFO] Booting up {self.model_name} via Ollama...")
        final_insights = {}

        for cluster_name, data in self.clusters_data.items():
            ticket_count = data['ticket_count']
            sample_tickets = data['sample_tickets']
            tickets_to_analyze = sample_tickets[:10]
            
            print(f"\n[INFO] Analyzing {cluster_name} ({ticket_count} total tickets)...")
            
            system_prompt = """
            You are an expert B2B product analyst. Your job is to read a group of customer support tickets 
            that a machine learning algorithm has clustered together. 
            
            Identify the core issue and respond STRICTLY in this format:
            Title: [A short, 3-5 word name for the issue]
            Summary: [A concise 1-sentence summary of what is happening]
            Actionable Advice: [One step the product engineering team should take to fix it]
            """
            
            user_prompt = f"Here are the tickets to analyze:\n{json.dumps(tickets_to_analyze, indent=2)}"

            try:
                response = ollama.chat(model=self.model_name, messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ])
                
                llm_output = response['message']['content']
                print(f"\n--- {cluster_name} Insight ---")
                print(llm_output)
                print("-" * 40)
                
                final_insights[cluster_name] = {
                    "ticket_count": ticket_count,
                    "llm_analysis": llm_output
                }
                
            except Exception as e:
                print(f"[ERROR] Failed to communicate with Ollama on {cluster_name}: {e}")

        self._save_results(final_insights)

    def _save_results(self, final_insights):
        """Saves the completed analysis to a fresh JSON file."""
        os.makedirs(os.path.dirname(self.output_file), exist_ok=True)
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(final_insights, f, indent=4)
            
        print(f"\n[INFO] Pipeline Phase 3 Complete! Insights securely saved to {self.output_file}")


# --- Execution Step (Run in the same cell!) ---
insight_engine = B2BInsightGenerator(model_name="llama3:latest")

if insight_engine.load_data():
    insight_engine.generate_insights()

[INFO] Loading data from data/llm_ready_clusters.json...
[INFO] Successfully loaded 2 clusters for analysis.
[INFO] Booting up llama3:latest via Ollama...

[INFO] Analyzing Cluster_0 (17 total tickets)...

--- Cluster_0 Insight ---
Based on the customer support tickets provided, here's my analysis:

Title: Product Hardware Issues
Summary: Customers are experiencing strange noises and functionality problems with their purchased products, suspecting a hardware issue.
Actionable Advice: Conduct thorough testing to identify the root cause of the hardware issue, potentially including diagnostic tests or troubleshooting procedures, and provide a clear plan for resolution (repair, replacement, or refund) in each case.
----------------------------------------

[INFO] Analyzing Cluster_1 (33 total tickets)...

--- Cluster_1 Insight ---
Based on the customer support tickets, I identified the core issue and responded in the following format:

**Ticket 1**
Title: Product Configuration Issue
Summar